# Exploratory Data Analysis of Diabetes Hospital Encounters
### SWYNEX Technologies — Task 2: Exploratory Data Analysis (EDA)
**Author:** Abhijeet Arun Gore  
**Role:** Data Science Intern at SWYNEX Technologies  
**Dataset:** Diabetes 130-US Hospitals (1999–2008) — Cleaned Inpatient Dataset from Task 1  
**Repository:** [SWYNEX-Data-Preparation](https://github.com/AbhijeetArunGore/SWYNEX-Data-Preparation)  

---

## Section 1 — Title and Objective

### Purpose of Exploratory Data Analysis (EDA)
Exploratory Data Analysis is a foundational discipline in clinical data science. Its goal is not simply to plot distributions or calculate summary statistics, but to uncover the underlying data-generating mechanisms, detect data quality anomalies, validate empirical assumptions, test clinical hypotheses, and identify structural constraints prior to statistical or machine learning modeling.

### Historical Hospital Context
This study investigates a standardized cohort of **101,763 inpatient hospital encounters** occurring between **1999 and 2008** across **130 hospital facilities** in the United States. Each observation corresponds to an inpatient admission of an adult diabetic patient where hospital length of stay ranged from 1 to 14 days, and laboratory tests and pharmaceutical treatments were administered.

### Analytical and Business Questions
1. **Target Outcome Prevalence & Imbalance**: What is the baseline distribution of hospital readmissions, specifically within 30 days (`<30`), and what degree of class imbalance exists?
2. **Prior Utilization Impact**: Does prior healthcare utilization (inpatient admissions, emergency room visits, and outpatient encounters in the preceding 12 months) correlate with early readmission risk?
3. **Clinical Complexity & Acuity**: How do inpatient duration (`time_in_hospital`), pharmacological burden (`num_medications`), and diagnostic volume (`number_diagnoses`) relate to post-discharge stability?
4. **Discharge Planning & Transitions**: How do post-acute disposition destinations (e.g., Skilled Nursing Facilities, Rehabilitation Facilities, Home Health) compare in readmission risk?
5. **Glycemic Monitoring Adherence**: Does the documented performance of glycemic control testing (HbA1c tests) correlate with patient outcomes?
6. **Data Leakage & Clustering Risks**: Does the presence of 30,248 repeat admissions across 71,515 distinct patients introduce intra-subject correlation, and which terminal discharge events must be filtered prior to predictive modeling?

### Descriptive Analysis vs. Predictive Modeling
**Critical Methodological Distinction**: Task 2 is strictly exploratory and descriptive. Associations identified in this observational dataset **do not establish clinical causation**, nor do they guarantee predictive utility. Task 2 establishes the empirical evidence baseline and anti-leakage rules required for future Task 3 feature engineering and machine learning model development.


---
## Section 2 — Imports and Configuration
We initialize essential scientific computing, visualization, and statistical libraries, setting global parameters for reproducibility and professional reporting aesthetics.


In [ ]:
import os
import sys
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

# Add project root to path for modular utilities
sys.path.insert(0, os.path.abspath('..'))
from src.eda import (
    profile_dataset,
    compute_numerical_summary,
    compute_categorical_summary,
    compute_bivariate_rates,
    group_continuous_stats,
    perform_mann_whitney_u,
    perform_chi_square,
    compute_correlation_matrix
)

# Configure styling aesthetics
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams['font.sans-serif'] = 'Segoe UI'
plt.rcParams['figure.dpi'] = 120
plt.rcParams['axes.edgecolor'] = '#cccccc'
plt.rcParams['axes.linewidth'] = 0.8

# Global display and seed options
pd.set_option('display.max_columns', 60)
pd.set_option('display.max_rows', 100)
pd.set_option('display.width', 1000)
pd.set_option('display.float_format', lambda x: f'{x:.3f}' if abs(x) < 0.01 else f'{x:.2f}')
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print("Environment initialized: Scientific packages loaded, styling active, random seed = 42.")


---
## Section 3 — Load and Validate the Prepared Dataset
We load the exact cleaned clinical dataset exported in Task 1 (`data/processed/diabetic_data_clean.csv`) and execute comprehensive data-integrity checks comparing actual runtime properties with Task 1 output specifications.


In [ ]:
# Robust relative path handling
possible_paths = [
    os.path.join('data', 'processed', 'diabetic_data_clean.csv'),
    os.path.join('..', 'data', 'processed', 'diabetic_data_clean.csv')
]

data_path = None
for p in possible_paths:
    if os.path.exists(p):
        data_path = p
        break

if data_path is None:
    raise FileNotFoundError("Cleaned dataset not found in expected repository paths. Ensure Task 1 export exists.")

df = pd.read_csv(data_path)
profile = profile_dataset(df)

print("=== DATASET VALIDATION REPORT ===")
print(f"File Path: {data_path}")
print(f"Total Rows: {profile['total_rows']:,}")
print(f"Total Columns: {profile['total_columns']}")
print(f"Memory Footprint: {profile['memory_mb']} MB")
print(f"Exact Duplicate Rows: {profile['exact_duplicates']}")
print(f"Unique Patients: {profile['unique_patients']:,}")
print(f"Repeat Admission Percentage: {profile['repeat_admission_pct']}%")
print(f"Total Unhandled Null Values: {profile['total_null_values']}")

# Assertions for integrity
assert profile['total_rows'] == 101763, f"Expected 101,763 rows, found {profile['total_rows']}"
assert profile['total_null_values'] == 0, f"Found {profile['total_null_values']} unhandled nulls"
assert 'readmitted' in df.columns, "Target column 'readmitted' is missing!"
print("\nIntegrity Check: PASSED. Dataset strictly conforms to Task 1 specifications.")


In [ ]:
# Display sample records without exposing full patient identities
sample_cols = ['encounter_id', 'age', 'gender', 'race', 'time_in_hospital', 'num_medications', 'number_inpatient', 'diag_1_category', 'readmitted']
print("=== SAMPLE CLINICAL ENCOUNTERS ===")
df[sample_cols].head(5)


---
## Section 4 — Data Dictionary and Variable Groups
To structure our exploratory analysis, we categorize the 54 features of the prepared dataset into seven functional clinical and administrative domains. This prevents misleading analytical treatments—such as calculating continuous averages for relational keys or treating record IDs as clinical predictors.

| Functional Domain | Key Variables Included | Data Types | Analytical Role & Modeling Treatment |
| :--- | :--- | :--- | :--- |
| **Identifiers** | `encounter_id`, `patient_nbr` | Integer / Object | Primary keys. `encounter_id` identifies the unit of observation; `patient_nbr` is the patient-level grouping key for leak-free cross-validation. Never used as predictive features. |
| **Demographics** | `age`, `age_midpoint`, `gender`, `race` | Categorical / Float | Baseline patient attributes. `age_midpoint` converts 10-year brackets into continuous midpoints. |
| **Healthcare Utilization** | `number_inpatient`, `number_emergency`, `number_outpatient` | Integer Counts | Counts of prior encounters in the preceding 12 months. Extreme right skew; captures historical acute acuity and healthcare dependency. |
| **Inpatient Complexity** | `time_in_hospital`, `num_medications`, `num_procedures`, `num_lab_procedures`, `number_diagnoses` | Integer Counts | Measures of encounter resource intensity and multi-morbidity burden during the index admission. |
| **Diagnoses** | `diag_1`, `diag_2`, `diag_3`, `diag_1_category`, `diag_2_category`, `diag_3_category` | Categorical | Primary, secondary, and tertiary diagnoses mapped to 9 high-level clinical categories from ICD-9 codes. |
| **Administrative & Transfer** | `admission_type_desc`, `discharge_disposition_desc`, `admission_source_desc`, `payer_code`, `medical_specialty` | Categorical | Operational characteristics of hospital admission and discharge pathways. Contains key risk stratifiers and terminal status leakage indicators (`Expired`). |
| **Pharmacotherapy & Glycemia** | `max_glu_serum`, `A1Cresult`, `change`, `diabetesMed`, 23 antidiabetic drug columns | Categorical | Glycemic control laboratory results (`Not Tested`, `Norm`, `>7`, `>8`), overall medication change status, and individual medication titration levels. |
| **Target Outcome** | `readmitted` | Categorical (`NO`, `>30`, `<30`) | Primary clinical outcome. `<30` indicates early readmission within the benchmark 30-day window; `>30` represents late readmission; `NO` indicates no recorded readmission. |


In [ ]:
# Summary of feature distribution across functional domains
domain_mapping = {
    'Identifiers': ['encounter_id', 'patient_nbr'],
    'Demographics': ['age', 'age_midpoint', 'gender', 'race'],
    'Utilization': ['number_inpatient', 'number_emergency', 'number_outpatient'],
    'Inpatient Complexity': ['time_in_hospital', 'num_medications', 'num_procedures', 'num_lab_procedures', 'number_diagnoses'],
    'Diagnoses': ['diag_1', 'diag_2', 'diag_3', 'diag_1_category', 'diag_2_category', 'diag_3_category'],
    'Administrative & Transitions': ['admission_type_id', 'admission_type_desc', 'discharge_disposition_id', 'discharge_disposition_desc', 'admission_source_id', 'admission_source_desc', 'payer_code', 'medical_specialty'],
    'Pharmacotherapy & Glycemia': ['max_glu_serum', 'A1Cresult', 'change', 'diabetesMed'] + [col for col in df.columns if col in ['metformin', 'repaglinide', 'nateglinide', 'chlorpropamide', 'glimepiride', 'acetohexamide', 'glipizide', 'glyburide', 'tolbutamide', 'pioglitazone', 'rosiglitazone', 'acarbose', 'miglitol', 'troglitazone', 'tolazamide', 'insulin', 'glyburide-metformin', 'glipizide-metformin', 'glimepiride-pioglitazone', 'metformin-rosiglitazone', 'metformin-pioglitazone']],
    'Target Outcome': ['readmitted']
}

domain_summary = pd.DataFrame([
    {'Domain': k, 'Feature Count': len(v), 'Sample Features': ', '.join(v[:3]) + ('...' if len(v) > 3 else '')}
    for k, v in domain_mapping.items()
])

print("=== CLINICAL DOMAIN BREAKDOWN (54 FEATURES) ===")
domain_summary


---
## Section 5 — Descriptive Statistics

### Numerical Feature Analysis
We calculate the complete suite of central tendency, dispersion, and distribution asymmetry metrics across all numerical variables.

**Clinical Insight on Mean vs. Median**:
For symmetric variables such as `num_lab_procedures` (Mean = 43.10, Median = 44.0, Skew = -0.24), the mean provides a representative measure of clinical activity. Conversely, for severely right-skewed utilization metrics such as `number_inpatient` (Mean = 0.64, Median = 0.0, Skew = +3.61) and `number_emergency` (Mean = 0.20, Median = 0.0, Skew = +22.86), the mean is heavily distorted by rare frequent-utilizer outliers. In these variables, the **median and Interquartile Range (IQR)** represent the only mathematically sound summary statistics.


In [ ]:
num_cols = ['time_in_hospital', 'num_lab_procedures', 'num_procedures', 'num_medications',
            'number_outpatient', 'number_emergency', 'number_inpatient', 'number_diagnoses', 'age_midpoint']

num_summary = compute_numerical_summary(df, columns=num_cols)
print("=== COMPREHENSIVE NUMERICAL DESCRIPTIVE STATISTICS ===")
num_summary


### Categorical Feature Summary
We evaluate cardinalities, top frequency counts, and dominant categories across all demographic, administrative, and clinical features.


In [ ]:
cat_cols = ['race', 'gender', 'age', 'admission_type_desc', 'discharge_disposition_desc',
            'admission_source_desc', 'diag_1_category', 'max_glu_serum', 'A1Cresult', 'change', 'diabetesMed', 'readmitted']

cat_summary = compute_categorical_summary(df, columns=cat_cols)
print("=== CATEGORICAL FEATURE ATTRIBUTE FREQUENCIES ===")
cat_summary


---
## Section 6 — Univariate Analysis

We examine the marginal empirical distributions of key patient demographics, clinical complexity indices, and acute healthcare utilization metrics. Each visualization is paired with immediate analytical commentary and distribution metrics.


In [ ]:
# 1. Age Bracket Distribution
plt.figure(figsize=(10, 4.8))
age_order = ['[0-10)', '[10-20)', '[20-30)', '[30-40)', '[40-50)', '[50-60)', '[60-70)', '[70-80)', '[80-90)', '[90-100)']
age_counts = df['age'].value_counts().reindex(age_order)
age_pcts = (age_counts / len(df)) * 100

bar_plot = sns.barplot(x=age_counts.index, y=age_counts.values, palette='Blues_d', edgecolor='black', linewidth=0.5)
plt.title('Univariate Distribution of Patient Age Brackets (N=101,763)', fontsize=12, fontweight='bold', pad=12)
plt.xlabel('Age Bracket (Years)', fontsize=11, fontweight='semibold')
plt.ylabel('Hospital Encounters', fontsize=11, fontweight='semibold')
plt.ylim(0, 30000)

for p, pct in zip(bar_plot.patches, age_pcts):
    height = p.get_height()
    plt.text(p.get_x() + p.get_width()/2., height + 600, f'{pct:.1f}%', ha='center', va='bottom', fontsize=9, fontweight='bold')

plt.tight_layout()
plt.show()


**Interpretation**: The diabetic inpatient cohort is predominantly geriatric. Encounters among patients aged 60–90 constitute **64.6%** of all hospitalizations (70–80: 25.6%; 60–70: 22.1%; 80–90: 16.9%). Pediatric and adolescent diabetic admissions (<20 years) represent less than 0.9% of total encounters. Future predictive models must account for geriatric comorbidities and functional decline.

In [ ]:
# 2. Length of Stay (time_in_hospital)
fig, (ax_box, ax_hist) = plt.subplots(2, 1, figsize=(10, 5.5), gridspec_kw={'height_ratios': [0.25, 0.75]}, sharex=True)

sns.boxplot(x=df['time_in_hospital'], color='#2b5c8f', ax=ax_box)
ax_box.set(xlabel='')
ax_box.set_title('Distribution of Length of Hospital Stay (Days) [Median=4.0, Mean=4.40, IQR=4.0]', fontsize=12, fontweight='bold')

sns.histplot(df['time_in_hospital'], discrete=True, color='#2b5c8f', kde=True, ax=ax_hist)
ax_hist.set_xlabel('Time in Hospital (Days)', fontsize=11, fontweight='semibold')
ax_hist.set_ylabel('Encounter Frequency', fontsize=11, fontweight='semibold')

plt.tight_layout()
plt.show()


**Interpretation**: Length of hospital stay exhibits a moderate right skew (+1.13) with a median of 4.0 days and an interquartile range of 4.0 days (25th percentile = 2.0 days; 75th percentile = 6.0 days). Over 53% of hospitalizations resolve within 3 days, but a prolonged inpatient tail extends to the dataset maximum of 14 days.

In [ ]:
# 3. Primary Diagnosis Disease Categories (diag_1_category)
plt.figure(figsize=(10, 5))
diag_counts = df['diag_1_category'].value_counts()
diag_pcts = (diag_counts / len(df)) * 100

bars_d = sns.barplot(x=diag_counts.values, y=diag_counts.index, palette='crest_r', edgecolor='black', linewidth=0.5)
plt.title('Distribution of Primary Inpatient Diagnosis Categories (ICD-9)', fontsize=12, fontweight='bold', pad=12)
plt.xlabel('Encounter Count', fontsize=11, fontweight='semibold')
plt.ylabel('Primary Clinical Category', fontsize=11, fontweight='semibold')
plt.xlim(0, 36000)

for p, pct in zip(bars_d.patches, diag_pcts):
    width = p.get_width()
    plt.text(width + 500, p.get_y() + p.get_height()/2., f'{width:,} ({pct:.1f}%)', va='center', ha='left', fontsize=9, fontweight='bold')

plt.tight_layout()
plt.show()


**Interpretation**: Despite all patients having a recorded diagnosis of diabetes, the primary reason for acute hospital admission was **Circulatory Disease** in 29.9% of cases (30,436 encounters), followed by Respiratory Disease (14.2%) and Digestive conditions (9.3%). Primary acute diabetes admissions accounted for 8.6% of encounters (8,757), demonstrating that cardiovascular manifestations dominate diabetic inpatient morbidity.

---
## Section 7 — Missingness and Data Quality

### Analytical Implications of Clinical Missingness
Although Task 1 standardized data types and resolved all physical null entries (verifying **0 unhandled nulls** in the active dataset), EDA must rigorously investigate the *analytical and epidemiological meaning* of missingness. In real-world hospital informatics, missingness is rarely Missing Completely at Random (MCAR); instead, it frequently conveys informative missingness (Missing Not at Random — MNAR) reflecting clinical decision protocols, physician ordering habits, or patient socioeconomic barriers.

### Audit of Sentinel Categories Established in Task 1
1. **Administrative Missingness (`payer_code` & `medical_specialty`)**: In Task 1, 39.56% missing payer codes and 49.08% missing medical specialties were preserved under explicit categories (`'Missing'`). Uninsured and self-pay patients are less likely to have commercial payer codes assigned.
2. **Clinical Protocol Missingness (`max_glu_serum` & `A1Cresult`)**: Encoded as `'Not Tested'`. Missing lab results signify that the attending physician elected not to order a glycemic test during the encounter.
3. **Relational Null Lookups (`IDS_mapping.csv`)**: Missing or unmapped keys in discharge disposition or admission source were standardized to `'Not Available / Unmapped'` to prevent string parsing errors.


In [ ]:
# Compare 30-day readmission rates across missing/sentinel vs observed categories
sentinel_analysis = []

# 1. Payer Code: Missing vs Known
payer_missing = df[df['payer_code'] == 'Missing']['readmitted'].apply(lambda x: 1 if x == '<30' else 0)
payer_known = df[df['payer_code'] != 'Missing']['readmitted'].apply(lambda x: 1 if x == '<30' else 0)
sentinel_analysis.append({
    'Feature Domain': 'Payer Code',
    'Category Group': 'Missing / Unspecified',
    'Encounter Count': len(payer_missing),
    'Readmit_30d_Rate': round(payer_missing.mean() * 100, 2)
})
sentinel_analysis.append({
    'Feature Domain': 'Payer Code',
    'Category Group': 'Known Assigned Payer',
    'Encounter Count': len(payer_known),
    'Readmit_30d_Rate': round(payer_known.mean() * 100, 2)
})

# 2. Medical Specialty: Missing vs Known
spec_missing = df[df['medical_specialty'] == 'Missing']['readmitted'].apply(lambda x: 1 if x == '<30' else 0)
spec_known = df[df['medical_specialty'] != 'Missing']['readmitted'].apply(lambda x: 1 if x == '<30' else 0)
sentinel_analysis.append({
    'Feature Domain': 'Physician Specialty',
    'Category Group': 'Missing / Unassigned',
    'Encounter Count': len(spec_missing),
    'Readmit_30d_Rate': round(spec_missing.mean() * 100, 2)
})
sentinel_analysis.append({
    'Feature Domain': 'Physician Specialty',
    'Category Group': 'Specialist Assigned',
    'Encounter Count': len(spec_known),
    'Readmit_30d_Rate': round(spec_known.mean() * 100, 2)
})

# 3. HbA1c Lab Test: Not Tested vs Tested
a1c_untested = df[df['A1Cresult'] == 'Not Tested']['readmitted'].apply(lambda x: 1 if x == '<30' else 0)
a1c_tested = df[df['A1Cresult'] != 'Not Tested']['readmitted'].apply(lambda x: 1 if x == '<30' else 0)
sentinel_analysis.append({
    'Feature Domain': 'HbA1c Lab Test',
    'Category Group': 'Not Tested (No Lab Order)',
    'Encounter Count': len(a1c_untested),
    'Readmit_30d_Rate': round(a1c_untested.mean() * 100, 2)
})
sentinel_analysis.append({
    'Feature Domain': 'HbA1c Lab Test',
    'Category Group': 'Tested (Norm, >7, or >8)',
    'Encounter Count': len(a1c_tested),
    'Readmit_30d_Rate': round(a1c_tested.mean() * 100, 2)
})

sentinel_df = pd.DataFrame(sentinel_analysis)
print("=== SENTINEL AND INFORMATIVE MISSINGNESS RISK COMPARISON ===")
sentinel_df


### Visual Comparison of Sentinel Status vs Readmission Rate
We visualize how missing and unmeasured clinical attributes compare to measured attributes in 30-day readmission risk.


In [ ]:
plt.figure(figsize=(10, 4.8))
bar_s = sns.barplot(data=sentinel_df, x='Readmit_30d_Rate', y='Category Group', hue='Feature Domain', dodge=False, palette='Set2', edgecolor='black', linewidth=0.5)

plt.title('30-Day Readmission Risk Across Observed vs Informative Missingness Groups', fontsize=12, fontweight='bold', pad=12)
plt.xlabel('30-Day Readmission Rate (%)', fontsize=11, fontweight='semibold')
plt.ylabel('Clinical/Administrative Cohort', fontsize=11, fontweight='semibold')
plt.xlim(0, 15)

for p in bar_s.patches:
    width = p.get_width()
    if width > 0:
        plt.text(width + 0.3, p.get_y() + p.get_height()/2., f'{width:.2f}%', va='center', ha='left', fontsize=9.5, fontweight='bold')

plt.legend(title='Feature Domain', loc='lower right', frameon=True)
plt.tight_layout()
plt.show()


**Key Observation**: Patients without an ordered HbA1c test (`Not Tested`) had an **11.42%** 30-day readmission rate compared to **9.86%** for patients who received glycemic testing. Similarly, encounters with missing physician specialty information had a **10.59%** readmission rate versus **11.75%** for encounters where a specialist was actively assigned. This confirms that missingness reflects institutional workflow patterns and cannot be discarded without biasing the sample.

---
## Section 8 — Target/Outcome Analysis

### Outcome Variable Definitions & Empirical Distribution
In the prepared dataset, the primary outcome is recorded in the `readmitted` feature, containing three distinct clinical categories:
- **`NO`**: No recorded readmission within the historical encounter capture window.
- **`>30`**: Readmission occurred more than 30 days following the index hospital discharge.
- **`<30`**: Readmission occurred within 30 days of hospital discharge.

### Binary Classification Target Formulations for Future Modeling
When framing readmission prediction as a binary classification problem for Task 3, two competing definitions exist in health informatics literature:
1. **Early Readmission Benchmark (CMS Protocol)**: `<30` (Positive Class, 1) vs. `NO` and `>30` (Negative Class, 0). This models the 30-day readmission window monitored by the Centers for Medicare & Medicaid Services (CMS) Hospital Readmissions Reduction Program (HRRP).
2. **Any Recorded Readmission**: `<30` and `>30` (Positive Class, 1) vs. `NO` (Negative Class, 0). This models general patient recidivism and chronic disease instability.

**Methodological Warning**: These definitions are fundamentally distinct and must never be treated as equivalent. Definition A isolates acute post-discharge breakdown, whereas Definition B captures generalized chronic disease burden.


In [ ]:
# Compute exact outcome metrics
counts_3class = df['readmitted'].value_counts()[['NO', '>30', '<30']]
pcts_3class = (counts_3class / len(df)) * 100

df['readmit_30d'] = (df['readmitted'] == '<30').astype(int)
df['readmit_any'] = (df['readmitted'] != 'NO').astype(int)

target_summary = pd.DataFrame({
    'Readmission Category': counts_3class.index,
    'Encounter Count': counts_3class.values,
    'Cohort Percentage (%)': pcts_3class.values,
    'Clinical Definition': [
        'No readmission recorded in historical capture window',
        'Late readmission (>30 days post-discharge)',
        'Early readmission within CMS 30-day penalty threshold'
    ]
})

print("=== HOSPITAL READMISSION OUTCOME PREVALENCE ===")
target_summary


### Visualizing Class Distribution and Target Imbalance
We display the three-class outcome distribution alongside the binary 30-day early readmission formulation.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5.2))

# Left: 3-class distribution
bars1 = axes[0].bar(counts_3class.index, counts_3class.values, color=['#2b5c8f', '#e67e22', '#c0392b'], width=0.55, edgecolor='black', linewidth=0.5)
axes[0].set_title('A: Original 3-Class Readmission Distribution', fontsize=12, fontweight='bold', pad=12)
axes[0].set_xlabel('Readmission Category', fontsize=11, fontweight='semibold')
axes[0].set_ylabel('Hospital Encounters', fontsize=11, fontweight='semibold')
axes[0].set_ylim(0, 65000)

for bar, pct in zip(bars1, pcts_3class):
    y = bar.get_height()
    axes[0].text(bar.get_x() + bar.get_width()/2., y + 1200, f'{y:,}\n({pct:.2f}%)', ha='center', va='bottom', fontsize=9.5, fontweight='bold')

# Right: Binary 30-day distribution
bin_counts = pd.Series({
    'Not Readmitted <30 Days': len(df) - df['readmit_30d'].sum(),
    'Readmitted <30 Days (<30)': df['readmit_30d'].sum()
})
bin_pcts = (bin_counts / len(df)) * 100

bars2 = axes[1].bar(bin_counts.index, bin_counts.values, color=['#27ae60', '#c0392b'], width=0.45, edgecolor='black', linewidth=0.5)
axes[1].set_title('B: Binary CMS 30-Day Readmission Benchmark', fontsize=12, fontweight='bold', pad=12)
axes[1].set_xlabel('Classification Cohort', fontsize=11, fontweight='semibold')
axes[1].set_ylabel('Hospital Encounters', fontsize=11, fontweight='semibold')
axes[1].set_ylim(0, 105000)

for bar, pct in zip(bars2, bin_pcts):
    y = bar.get_height()
    axes[1].text(bar.get_x() + bar.get_width()/2., y + 1800, f'{y:,}\n({pct:.2f}%)', ha='center', va='bottom', fontsize=9.5, fontweight='bold')

plt.suptitle('Clinical Outcome Distribution & Severe Target Imbalance (N=101,763)', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()


**Analytical Finding on Class Imbalance**:
The primary target of interest (`<30`) occurs in only **11.16%** of encounters (11,357 cases). This reflects a **~8:1 negative-to-positive class imbalance**. Consequently:
1. **Accuracy is a Deceptive Metric**: A trivial null model that uniformly predicts "No 30-Day Readmission" yields an accuracy of **88.84%** while failing to identify a single vulnerable patient.
2. **Evaluation Strategy**: Model validation in Task 3 must prioritize **Precision-Recall AUC (PR-AUC)**, **Brier score calibration**, and **Recall at fixed decision thresholds** over raw accuracy or ROC-AUC alone.

---
## Section 9 — Bivariate Analysis and Clinical Relationships

We evaluate relationships between patient features and readmission outcomes, reporting normalized subgroup rates, sample denominators, and clinical implications.


In [ ]:
# 1. Readmission Risk vs. Prior Inpatient Utilization
inp_bivariate = compute_bivariate_rates(df, group_col='number_inpatient', target_col='readmitted')
print("=== PRIOR INPATIENT ADMISSIONS VS. READMISSION RATES ===")
inp_bivariate.head(8)


In [ ]:
# Plot prior inpatient visits vs. 30-day readmission rate
filtered_inp = inp_bivariate[inp_bivariate['Total_Encounters'] >= 100].copy()
filtered_inp['Inpatient_Visits'] = filtered_inp['Group_Value'].astype(int)
filtered_inp = filtered_inp.sort_values(by='Inpatient_Visits')

fig, ax1 = plt.subplots(figsize=(10, 5))
color_b = '#2b5c8f'
bars_i = ax1.bar(filtered_inp['Inpatient_Visits'], filtered_inp['Total_Encounters'], color=color_b, alpha=0.35, width=0.55, label='Encounter Count')
ax1.set_xlabel('Prior Inpatient Admissions in Preceding 12 Months', fontsize=11, fontweight='semibold')
ax1.set_ylabel('Total Encounters (Log Scale)', fontsize=11, fontweight='semibold', color=color_b)
ax1.set_yscale('log')
ax1.tick_params(axis='y', labelcolor=color_b)

ax2 = ax1.twinx()
color_r = '#c0392b'
ax2.plot(filtered_inp['Inpatient_Visits'], filtered_inp['Readmit_30d_Rate_Pct'], color=color_r, marker='o', linewidth=2.5, markersize=7, label='30-Day Readmission Rate (%)')
ax2.set_ylabel('30-Day Readmission Rate (%)', fontsize=11, fontweight='semibold', color=color_r)
ax2.tick_params(axis='y', labelcolor=color_r)
ax2.set_ylim(0, 42)

for x, y in zip(filtered_inp['Inpatient_Visits'], filtered_inp['Readmit_30d_Rate_Pct']):
    ax2.annotate(f'{y:.1f}%', (x, y), textcoords="offset points", xytext=(0, 10), ha='center', fontsize=9.5, fontweight='bold', color=color_r)

plt.title('Prior Inpatient Utilization as Chief Readmission Driver (8.44% to 34.58%)', fontsize=12, fontweight='bold', pad=12)
plt.tight_layout()
plt.show()


**Clinical Interpretation**: Prior inpatient hospitalization is the single most decisive predictor of readmission in this dataset. Patients with zero prior admissions exhibit an **8.44%** 30-day readmission rate. This escalates to **12.92%** at 1 visit, **20.29%** at 3 visits, and exceeds **34%** for chronic recidivists with 5+ visits. The step-change reflects severe underlying systemic illness and inadequate outpatient care stabilization.

In [ ]:
# 2. Discharge Disposition vs. Readmission Risk
disch_bivariate = compute_bivariate_rates(df, group_col='discharge_disposition_desc', target_col='readmitted', min_count=500)
print("=== TOP DISCHARGE DESTINATIONS VS. READMISSION RATES ===")
disch_bivariate.head(10)


In [ ]:
# Visualize readmission by discharge disposition
plt.figure(figsize=(11, 5.5))
sorted_disch = disch_bivariate.sort_values(by='Readmit_30d_Rate_Pct', ascending=True)

# Truncate lengthy labels for visual clarity
short_labels = [s[:45] + ('...' if len(s) > 45 else '') for s in sorted_disch['Group_Value']]
bars_d = plt.barh(short_labels, sorted_disch['Readmit_30d_Rate_Pct'], color='#2b5c8f', edgecolor='black', linewidth=0.5)

plt.title('30-Day Readmission Rate by Primary Discharge Disposition Destination', fontsize=12, fontweight='bold', pad=12)
plt.xlabel('30-Day Readmission Rate (%)', fontsize=11, fontweight='semibold')
plt.ylabel('Discharge Destination Category', fontsize=11, fontweight='semibold')
plt.xlim(0, 32)

for bar, n in zip(bars_d, sorted_disch['Total_Encounters']):
    w = bar.get_width()
    plt.text(w + 0.5, bar.get_y() + bar.get_height()/2., f'{w:.2f}% (N={n:,})', va='center', ha='left', fontsize=9, fontweight='bold')

plt.tight_layout()
plt.show()


**Clinical Interpretation & Data Leakage Caveat**:
1. **Post-Acute Transitions**: Transfers to Rehabilitation Facilities (**27.71%**) and Skilled Nursing Facilities (**14.66%**) suffer substantially worse 30-day readmission rates than home discharges (**9.30%**), reflecting patient frailty and complex post-discharge care demands.
2. **Terminal Status Target Leakage**: 1,642 encounters resulted in in-hospital death (`Expired`), all showing an exact **0.00%** readmission rate. Patients who expire during their stay are physically incapable of post-discharge readmission. In Task 3, **all expired records must be excluded prior to training** to prevent the model from learning artificial shortcuts.

In [ ]:
# 3. Clinical Intensity: Length of Stay & Medications by Readmission Outcome
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Boxplot of Time in Hospital
sns.boxplot(data=df, x='readmitted', y='time_in_hospital', order=['NO', '>30', '<30'], palette=['#2b5c8f', '#e67e22', '#c0392b'], ax=axes[0])
axes[0].set_title('A: Length of Hospital Stay by Readmission Group', fontsize=11, fontweight='bold')
axes[0].set_xlabel('Readmission Outcome', fontsize=10, fontweight='semibold')
axes[0].set_ylabel('Time in Hospital (Days)', fontsize=10, fontweight='semibold')

# Boxplot of Number of Medications
sns.boxplot(data=df, x='readmitted', y='num_medications', order=['NO', '>30', '<30'], palette=['#2b5c8f', '#e67e22', '#c0392b'], ax=axes[1])
axes[1].set_title('B: Medication Count by Readmission Group', fontsize=11, fontweight='bold')
axes[1].set_xlabel('Readmission Outcome', fontsize=10, fontweight='semibold')
axes[1].set_ylabel('Number of Medications', fontsize=10, fontweight='semibold')

plt.suptitle('Comparison of Inpatient Complexity Metrics Across Readmission Cohorts', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()


**Clinical Interpretation**: Patients readmitted within 30 days had longer median hospital stays (4.0 days; mean = 4.77) and higher median medication counts (15.0; mean = 16.71) compared to patients not readmitted (median stay = 3.0 days; mean = 4.18; mean medications = 15.67). While statistically significant, the overlap in interquartile ranges demonstrates that length of stay alone cannot serve as an isolated risk discriminator.

---
## Section 10 — Statistical Tests, Where Justified

### Hypothesis Testing Governance Framework
To test whether observed differences between patient subgroups are statistically robust or attributable to random variation, we apply formal non-parametric and contingency hypothesis tests.

**Important Methodological Limitation on Sample Independence**:
The dataset contains **101,763 encounters** across **71,515 distinct patients**, with **29.72%** being repeat hospitalizations. Consequently, observations from the same individual are not strictly independent (intra-cluster correlation). While large sample sizes generate very small p-values ($p < 10^{-15}$), these tests should be viewed as descriptive indicators of distributional separation rather than definitive clinical proof.

---

### Test 1: Prior Inpatient Admissions across Readmission Outcomes (Mann-Whitney U)
- **$H_0$**: The distribution of prior inpatient visits is identical between encounters readmitted within 30 days (`<30`) and those not readmitted (`NO`).
- **$H_1$**: The distribution of prior inpatient visits for the `<30` group is stochastically shifted toward higher values.
- **Assumptions**: Continuous/discrete ordinal scale, independent cohorts, no normality assumption required.


In [ ]:
# Execute Mann-Whitney U test on prior inpatient admissions
test1_res = perform_mann_whitney_u(df, continuous_col='number_inpatient', group_col='readmitted', group_val1='<30', group_val2='NO')
print("=== STATISTICAL TEST 1 RESULTS: PRIOR INPATIENT ADMISSIONS ===")
for k, v in test1_res.items():
    print(f"  {k:28s}: {v}")


---
### Test 2: Length of Hospital Stay across Readmission Outcomes (Mann-Whitney U)
- **$H_0$**: The distribution of length of hospital stay (`time_in_hospital`) is identical between `<30` and `NO` cohorts.
- **$H_1$**: Patients readmitted within 30 days experience stochastically longer hospital stays.


In [ ]:
# Execute Mann-Whitney U test on length of stay
test2_res = perform_mann_whitney_u(df, continuous_col='time_in_hospital', group_col='readmitted', group_val1='<30', group_val2='NO')
print("=== STATISTICAL TEST 2 RESULTS: LENGTH OF HOSPITAL STAY ===")
for k, v in test2_res.items():
    print(f"  {k:28s}: {v}")


---
### Categorical Association Testing: Pearson Chi-Square & Cramer's V Effect Size
We evaluate categorical associations between readmission outcome (`readmitted`) and key clinical features: patient age bracket, diabetes medication prescription, and in-hospital medication dosage change.


In [ ]:
# Execute Chi-Square tests
chi_tests = [
    perform_chi_square(df, cat_col1='age', cat_col2='readmitted'),
    perform_chi_square(df, cat_col1='diabetesMed', cat_col2='readmitted'),
    perform_chi_square(df, cat_col1='change', cat_col2='readmitted'),
    perform_chi_square(df, cat_col1='diag_1_category', cat_col2='readmitted')
]

chi_summary_df = pd.DataFrame(chi_tests)
print("=== CHI-SQUARE INDEPENDENCE TESTS & EFFECT SIZES ===")
chi_summary_df[['variable_1', 'chi2_statistic', 'degrees_of_freedom', 'p_value', 'cramers_v', 'statistically_significant']]


**Statistical Interpretation**:
1. **Prior Inpatient Admissions**: The Mann-Whitney U test yields $U = 401,064,764, p < 10^{-15}$. Encounters in the `<30` group averaged 1.224 prior inpatient admissions compared to 0.382 in the `NO` cohort (over 3.2x higher).
2. **Length of Stay**: The Mann-Whitney U test yields $U = 347,113,746, p = 1.595 	imes 10^{-83}$, confirming that readmitted patients have a statistically significant right-shifted hospital duration (median = 4.0 vs 3.0 days).
3. **Medication Adjustments & Regimens**: Chi-Square tests demonstrate significant dependency between readmission and medication change ($\chi^2 = 215.94, p = 1.289 	imes 10^{-47}$) as well as diabetes medication prescription ($\chi^2 = 386.44, p = 1.218 	imes 10^{-84}$). Patients on active anti-diabetic medications experienced higher readmission rates (11.58%) than those on no diabetes medications (8.01%).